---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group XX

- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

*italicised text*## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [17]:
%pip install numpy pandas matplotlib qpsolvers osqp

Note: you may need to restart the kernel to use updated packages.


## Import Libraries

Every import of the notebook is kept in this cell.

In [67]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import sys
from IPython.display import display
import matplotlib.pyplot as plt

## Settings

Paths and shared constants are kept in this class.

In [70]:
class Settings:
    ROOT = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "src" / "notebook.ipynb").is_file() and (path / "dataset").is_dir()),
        None,
    )
    if ROOT is None:
        raise FileNotFoundError("Start the notebook from the repository root or src/.")

    SEED = 42
    DATA_DIR = ROOT / "dataset"
    OUTPUT_DIR = Path(os.environ.get("MAIN_OUTPUT_DIR", ROOT / "output"))
    TARGET = "is_canceled"
    ID = "id"

CFG = Settings()
if str(CFG.ROOT) not in sys.path:
    sys.path.insert(0, str(CFG.ROOT))
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Import Algoritma Local Repo
from src.decision_tree import DecisionTree
from src.decision_tree.optimized.decision_tree import DecisionTree as OptimizedTree
from src.decision_tree.tree_visualization import plot_decision_tree
from src.logistic_regression.logistic_regression import LogisticRegression
from src.logistic_regression.optimized import LROptimized
from src.svm.svm import SVM

## Seed Everything

A fixed seed keeps the results reproducible.

In [71]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    return np.random.default_rng(seed)

split_rng = seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [72]:
train_df = pd.read_csv(CFG.DATA_DIR / "train.csv", dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / "test.csv", dtype={'id': str})


print(f'Train: {train_df.shape}  Test: {test_df.shape}')

Train: (27373, 19)  Test: (6844, 18)


---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [22]:
# Write your code here

## Explore Data

This part covers distributions and relationships between attributes.

In [23]:
# Write your code here

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [24]:
# Write your code here

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [73]:
labels = train_df[CFG.TARGET].to_numpy()
train_indices, val_indices = [], []

for label in np.unique(labels):
    indices = split_rng.permutation(np.flatnonzero(labels == label))
    n_val = round(0.20 * len(indices))
    val_indices.append(indices[:n_val])
    train_indices.append(indices[n_val:])

train_set = train_df.iloc[split_rng.permutation(np.concatenate(train_indices))]
val_set = train_df.iloc[split_rng.permutation(np.concatenate(val_indices))]

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [74]:
X_train = train_set.drop(columns=[CFG.ID, CFG.TARGET]).copy()
X_val = val_set.drop(columns=[CFG.ID, CFG.TARGET]).copy()
y_train = train_set[CFG.TARGET].copy()
y_val = val_set[CFG.TARGET].copy()

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [75]:
def fill_missing(frame, medians, missing_columns, categorical_columns):
    frame = frame.copy()
    frame[missing_columns + "_missing"] = frame[missing_columns].isna().astype(float).to_numpy()
    frame[medians.index] = frame[medians.index].fillna(medians)
    frame[categorical_columns] = frame[categorical_columns].fillna("<missing>")
    return frame

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.columns.difference(numeric_columns, sort=False).tolist()
missing_columns = X_train[numeric_columns].columns[X_train[numeric_columns].isna().any()]
numeric_medians = X_train[numeric_columns].median().fillna(0.0)

X_train = fill_missing(X_train, numeric_medians, missing_columns, categorical_columns)
X_val = fill_missing(X_val, numeric_medians, missing_columns, categorical_columns)

### Duplicates

This part covers repeated records.

In [76]:
display(pd.Series({
    "train_duplicate_rows": train_set.duplicated().sum(),
    "validation_duplicate_rows": val_set.duplicated().sum(),
    "train_repeated_predictors": train_set.drop(columns=[CFG.ID, CFG.TARGET]).duplicated().sum(),
}, name="count"))

train_duplicate_rows          0
validation_duplicate_rows     0
train_repeated_predictors    40
Name: count, dtype: int64

### Outliers

This part covers values far from the rest of the data.

In [77]:
display(X_train[numeric_columns].describe(percentiles=[0.01, 0.5, 0.99]).T[
    ["min", "1%", "50%", "99%", "max"]
])

,min,1%,50%,99%,max
num_adults,0.0,1.0,2.0,3.0,4.0
num_children,0.0,0.0,0.0,2.0,10.0
weekend_nights,0.0,0.0,1.0,2.0,7.0
weekday_nights,0.0,0.0,2.0,6.0,17.0
needs_parking,0.0,0.0,0.0,1.0,1.0
days_before_arrival,0.0,0.0,58.0,343.0,448.0
arrival_year,2023.0,2023.0,2024.0,2024.0,2024.0
arrival_month,1.0,1.0,8.0,12.0,12.0
arrival_day,1.0,1.0,15.0,31.0,31.0
is_returning_guest,0.0,0.0,0.0,1.0,1.0


### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [30]:
# Write your code here

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

This part covers new attributes derived from existing ones.

In [78]:
def construct_features(frame):
    return frame.assign(total_nights=frame["weekday_nights"] + frame["weekend_nights"])

X_train = construct_features(X_train)
X_val = construct_features(X_val)
numeric_columns = numeric_columns + ["total_nights"]

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [79]:
def scale_features(frame, means, scales):
    frame = frame.copy()
    frame[means.index] = (frame[means.index] - means) / scales
    return frame

numeric_means = X_train[numeric_columns].mean()
numeric_scales = X_train[numeric_columns].std(ddof=0).replace(0, 1.0)

X_train = scale_features(X_train, numeric_means, numeric_scales)
X_val = scale_features(X_val, numeric_means, numeric_scales)

### Encoding

This part covers the numeric representation of categorical attributes.

In [80]:
def encode_features(frame, levels):
    frame = frame.assign(**{
        column: pd.Categorical(frame[column], categories=categories)
        for column, categories in levels.items()
    })
    return pd.get_dummies(frame, columns=list(levels), dtype=float)

category_levels = {
    column: sorted(set(X_train[column]) | {"<missing>"})
    for column in categorical_columns
}
X_train = encode_features(X_train, category_levels)
X_val = encode_features(X_val, category_levels)

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [34]:
# Write your code here


## Class Balance

This part covers the proportion of each class in the training data.

In [81]:
display(pd.DataFrame({
    "count": y_train.value_counts(),
    "share": y_train.value_counts(normalize=True),
}).sort_index())


,count,share
is_canceled,,
0,14723,0.672345
1,7175,0.327655


## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [82]:
X_train = X_train.astype(float)
X_val = X_val.astype(float)
y_train = y_train.astype(int)
y_val = y_val.astype(int)
print(f"Training: {X_train.shape}\nvalidation: {X_val.shape}")

Training: (21898, 40)
validation: (5475, 40)


---

# Preparation Pipeline <a name="6"></a>

---

The pipeline combines the preparation steps so they run in one consistent order.

In [83]:
def fit_preparation(raw):
    numeric = raw.select_dtypes(include="number")
    categorical = raw.columns.difference(numeric.columns, sort=False).tolist()
    medians = numeric.median().fillna(0.0)
    missing = numeric.columns[numeric.isna().any()]
    frame = construct_features(fill_missing(raw, medians, missing, categorical))
    numeric_columns = numeric.columns.tolist() + ["total_nights"]
    return {
        "features": raw.columns.tolist(),
        "medians": medians,
        "missing_columns": missing,
        "categorical_columns": categorical,
        "means": frame[numeric_columns].mean(),
        "scales": frame[numeric_columns].std(ddof=0).replace(0, 1.0),
        "categories": {
            column: sorted(set(frame[column]) | {"<missing>"})
            for column in categorical
        },
    }

def prepare(raw, state):
    frame = fill_missing(
        raw[state["features"]], state["medians"],
        state["missing_columns"], state["categorical_columns"],
    )
    frame = construct_features(frame)
    frame = scale_features(frame, state["means"], state["scales"])
    return encode_features(frame, state["categories"]).astype(float)

X_train_raw = train_set.drop(columns=[CFG.ID, CFG.TARGET])
X_val_raw = val_set.drop(columns=[CFG.ID, CFG.TARGET])
X_test_raw = test_df.drop(columns=CFG.ID)

preparation = fit_preparation(X_train_raw)
X_train = prepare(X_train_raw, preparation)
X_val = prepare(X_val_raw, preparation)
X_test = prepare(X_test_raw, preparation)
print(f"Prepared matrices: train {X_train.shape}, validation {X_val.shape}, test {X_test.shape}")

Prepared matrices: train (21898, 40), validation (5475, 40), test (6844, 40)


---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

This part covers the nearest neighbor classifier and its validation.

In [38]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [84]:
fitted_models = {}
valid_predictions = {}

tree_params = dict(max_depth=10, min_samples_leaf=20, random_state=CFG.SEED)
for name, model_class in (
    ("Decision tree", DecisionTree),
    ("Optimized decision tree", OptimizedTree),
):
    model = model_class(**tree_params).fit(X_train, y_train)
    predictions = model.predict(X_val)
    fitted_models[name] = model
    valid_predictions[name] = predictions
    print(f"{name}: validation accuracy = {np.mean(predictions == y_val.to_numpy()):.4f}")

Decision tree: validation accuracy = 0.8468
Optimized decision tree: validation accuracy = 0.8493


## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [85]:
lr_candidates = {
    "Logistic regression (L2)": (LogisticRegression, dict(
        penalty="l2", alpha=0.001, batch_size=256, max_iter=100, random_state=CFG.SEED,
    )),
}
for penalty in ("l1", "l2", "elasticnet"):
    lr_candidates[f"Optimized logistic regression ({penalty})"] = (LROptimized, dict(
        penalty=penalty, alpha=0.001, max_iter=100, random_state=CFG.SEED,
    ))

for name, (model_class, params) in lr_candidates.items():
    model = model_class(**params).fit(X_train, y_train)
    predictions = model.predict(X_val)
    fitted_models[name] = model
    valid_predictions[name] = predictions
    print(f"{name}: validation accuracy = {np.mean(predictions == y_val.to_numpy()):.4f}")

Logistic regression (L2): validation accuracy = 0.8027
Optimized logistic regression (l1): validation accuracy = 0.8015
Optimized logistic regression (l2): validation accuracy = 0.8013
Optimized logistic regression (elasticnet): validation accuracy = 0.8015


## Support Vector Machine

This part covers the linear, polynomial, and Gaussian kernels.

In [86]:
SVM_TRAIN_LIMIT = 1000
svm_rng = np.random.default_rng(CFG.SEED)
svm_labels = y_train.to_numpy()
svm_limit = min(SVM_TRAIN_LIMIT, len(svm_labels))
kept = np.flatnonzero(svm_labels == 0)
canceled = np.flatnonzero(svm_labels == 1)
n_canceled = round(svm_limit * len(canceled) / len(svm_labels))

svm_indices = np.concatenate([
    svm_rng.choice(kept, svm_limit - n_canceled, replace=False),
    svm_rng.choice(canceled, n_canceled, replace=False),
])
X_svm = X_train.to_numpy()[svm_indices]
y_svm = 2 * svm_labels[svm_indices] - 1
X_val_svm = X_val.to_numpy()
gamma = 1.0 / X_train.shape[1]

svm_params = {
    "SVM (linear)": dict(kernel="linear", c=1.0),
    "SVM (polynomial)": dict(kernel="polynomial", c=1.0, gamma=gamma, r=1.0, p=2),
    "SVM (gaussian)": dict(kernel="gaussian", c=1.0, gamma=gamma),
}
for name, params in svm_params.items():
    model = SVM(**params).fit(X_svm, y_svm)
    predictions = np.array([(model.predict(row) + 1) // 2 for row in X_val_svm], dtype=int)
    fitted_models[name] = model
    valid_predictions[name] = predictions
    print(f"{name}: {svm_limit:,} training rows, validation accuracy = {np.mean(predictions == y_val.to_numpy()):.4f}")

/home/leains/Desktop/ITB/SEM_5/AI/TUBES/IF3170_G22_OISCDivisiKaggle/venv/lib/python3.14/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:28: SparseConversionWarning: Converted matrix 'P' of your problem to scipy.sparse.csc_matrix to pass it to solver 'osqp'; for best performance, build your matrix as a csc_matrix directly.
  warnings.warn(
/home/leains/Desktop/ITB/SEM_5/AI/TUBES/IF3170_G22_OISCDivisiKaggle/venv/lib/python3.14/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:28: SparseConversionWarning: Converted matrix 'A' of your problem to scipy.sparse.csc_matrix to pass it to solver 'osqp'; for best performance, build your matrix as a csc_matrix directly.
  warnings.warn(


SVM (linear): 1,000 training rows, validation accuracy = 0.7938
SVM (polynomial): 1,000 training rows, validation accuracy = 0.7982
SVM (gaussian): 1,000 training rows, validation accuracy = 0.7945


## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [42]:
# Write your code here

## Model Evaluation

This part compares the validation results of all models.

In [43]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [44]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [45]:
# Write your code here

---

# Saving and Loading Models <a name="9"></a>

---

This section stores the trained models and restores them for prediction.

In [46]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [87]:
validation_scores = []
actual = y_val.to_numpy()
for name, predicted in valid_predictions.items():
    tp = np.sum((actual == 1) & (predicted == 1))
    fp = np.sum((actual == 0) & (predicted == 1))
    fn = np.sum((actual == 1) & (predicted == 0))
    denominator = 2 * tp + fp + fn
    validation_scores.append({
        "model": name,
        "f1": 2 * tp / denominator if denominator else 0.0,
        "accuracy": np.mean(actual == predicted),
    })
submission_scores = pd.DataFrame(validation_scores).set_index("model").sort_values(
    ["f1", "accuracy"], ascending=False, kind="stable",
)
winner = submission_scores.index[0]

full_raw = train_df.drop(columns=[CFG.ID, CFG.TARGET])
full_y = train_df[CFG.TARGET].to_numpy(dtype=int)
final_preparation = fit_preparation(full_raw)
X_full = prepare(full_raw, final_preparation)
X_test_final = prepare(test_df.drop(columns=CFG.ID), final_preparation)

if winner in svm_params:
    final_params = svm_params[winner]
elif winner in lr_candidates:
    final_params = lr_candidates[winner][1]
else:
    final_params = tree_params
final_model = type(fitted_models[winner])(**final_params)

if winner in svm_params:
    rng = np.random.default_rng(CFG.SEED)
    limit = min(SVM_TRAIN_LIMIT, len(full_y))
    canceled = np.flatnonzero(full_y == 1)
    kept = np.flatnonzero(full_y == 0)
    n_canceled = round(limit * len(canceled) / len(full_y))
    indices = np.concatenate([
        rng.choice(kept, limit - n_canceled, replace=False),
        rng.choice(canceled, n_canceled, replace=False),
    ])
    final_model.fit(X_full.to_numpy()[indices], 2 * full_y[indices] - 1)
    test_predictions = np.array([
        (final_model.predict(row) + 1) // 2 for row in X_test_final.to_numpy()
    ], dtype=int)
else:
    final_model.fit(X_full, full_y)
    test_predictions = final_model.predict(X_test_final)

sample = pd.read_csv(CFG.DATA_DIR / "sample_submission.csv", dtype={CFG.ID: str})
if (not test_df[CFG.ID].is_unique or not sample[CFG.ID].is_unique or set(test_df[CFG.ID]) != set(sample[CFG.ID])):
    raise ValueError("Test IDs and sample_submission IDs must match and be unique.")

prediction_by_id = pd.Series(test_predictions, index=test_df[CFG.ID])
submission = sample[[CFG.ID]].assign(**{CFG.TARGET: sample[CFG.ID].map(prediction_by_id)})
if not submission[CFG.TARGET].isin([0, 1]).all():
    raise ValueError("Submission contains missing or invalid predictions.")

submission[CFG.TARGET] = submission[CFG.TARGET].astype(int)

submission_path = CFG.OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Selected model: {winner}")
print(f"Wrote {len(submission):,} predictions to {submission_path}")
display(submission.head())

Selected model: Decision tree
Wrote 6,844 predictions to /home/leains/Desktop/ITB/SEM_5/AI/TUBES/IF3170_G22_OISCDivisiKaggle/output/submission.csv


,id,is_canceled
0,5cb6f473,0
1,c06b4ab3,1
2,22bf3fc3,0
3,25408a48,0
4,621586b6,0


---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.